# Operational Risk: OpVaR from a Simulated Annual-Loss Distribution

Runs `operational_risk.operational_var_opvar` -- reads the loss quantile at a given confidence level directly from a supplied annual-loss sample (`engine/oprisk_lda.py`), the generic reader function that sits downstream of pyvar's LDA/compound-loss simulation. Requires a free-tier API key: https://www.pyvar.com#get-api-key

In [ ]:
%load_ext pyvar_jupyter
%pyvar_key eyJ...  # replace with your own key, or set PYVAR_API_KEY before starting the kernel

## A synthetic annual aggregate-loss sample

10,000 simulated years of aggregate operational loss, lognormally distributed -- a fixed seed so this is reproducible without a real loss database. In practice this array is `monte_carlo_oprisk_capital`'s own simulated-loss output, or a bank's internal loss data, not hand-generated like this.

In [ ]:
import numpy as np

rng = np.random.default_rng(23)
annual_losses = rng.lognormal(mean=13.0, sigma=1.1, size=10_000)
annual_losses[:5], annual_losses.size

## Run it at the Basel AMA confidence level (99.9%)

Line-magic form can't take a 10,000-element array inline -- use `client` directly instead of `%pyvar`:

In [ ]:
import os

from pyvar_client import Client

with Client(api_key=os.environ["PYVAR_API_KEY"]) as client:
    result = client.operational_risk.operational_var_opvar(
        annual_losses=annual_losses.tolist(), confidence_level=0.999
    )
result

With this seed, `opvar` comes out to roughly £12.9m at the 99.9th percentile of the 10,000 simulated years, `expected_shortfall` (the mean loss *beyond* that quantile) to roughly £23.8m, and `expected_loss` (the plain sample mean) to roughly £802k -- `expected_shortfall >= opvar >= expected_loss` always holds by construction, the same tail-ordering property `market_risk`'s ES/VaR pair guarantees, just applied to a loss distribution instead of a P&L distribution.